<a href="https://colab.research.google.com/github/piaseckazaneta/Python/blob/main/Location_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install h3 osmnx


import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import osmnx as ox #skrót od OpenStreetMap + NetworkX
from shapely.geometry import Polygon

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 4.0 MB/s eta 0:00:00


In [3]:
# 1. Pobieramy obrys miasta
place = "Warszawa, Poland"
city_boundary = ox.geocode_to_gdf(place)


In [4]:
# 2. Konwersja geometrii Shapely na komórki H3 (obsługuje Polygon i MultiPolygon)
geom = city_boundary.geometry.iloc[0]

# Wyciągamy poligony (jeśli miasto ma enklawy / jest MultiPolygonem)
polygons = [geom] if isinstance(geom, Polygon) else list(geom, geoms)

hex_ids = set()
for poly in polygons:
  # Shapely przechowuje (lng, lat) -> H3 v4 wymaga LatLngPoly ze współrzędnymi (lat, lng)
  outer_coords = [(lat, lng) for lng, lat in poly.exterior.coords]
  holes = [[(lat, lng) for lng, lat in hole.coords] for hole in poly.interiors]

  h3_poly = h3.LatLngPoly(outer_coords, holes)
  cells = h3.polygon_to_cells(h3_poly, res=9)
  hex_ids.update(cells)

hex_ids = list(hex_ids)

# 3. Konwersja komórek H3 z powrotem na GeoDataFrame (dla GeoPandas)
hex_polys = [
    Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(hid)])
    for hid in hex_ids
]
hex_gdf = gpd.GeoDataFrame({'hex_id': hex_ids, 'geometry': hex_polys}, crs="EPSG:4326")

print(f"Sukces! Utworzono {len(hex_gdf)} heksagonów H3 dla miasta {place}.")
hex_gdf.head(3)

Sukces! Utworzono 5331 heksagonów H3 dla miasta Warszawa, Poland.


,hex_id,geometry
0,891f53c8163ffff,"POLYGON ((21.09059 52.2679, 21.09053 52.26624,..."
1,891f5352557ffff,"POLYGON ((21.16832 52.22027, 21.16826 52.21861..."
2,891f53cb563ffff,"POLYGON ((20.98405 52.30312, 20.98399 52.30146..."


In [5]:
# 3: Definiujemy tagi OSM, które nas interesują
tags = {
    'amenity': ['cafe', 'restaurant', 'fast_food', 'supermarket'],
    'public_transport': 'platform',
    'highway': 'bus_stop'
}

In [6]:
print("Pobieranie POI z OpenStreetMap (może potrwać nawet 2 min)...")
pois_raw = ox.features_from_place(place, tags)
print(f"Pobrano łącznie {len(pois_raw)}")

Pobieranie POI z OpenStreetMap (może potrwać nawet 2 min)...
Pobrano łącznie 9261


In [7]:
# 4: Spłaszczenie geometrii do punktów (Centroid)
# W OSM budynek restauracji lub supermarketu bywa poligonem.
# Zamieniamy każdy obiekt na pojedynczy punkt środkowy (centroid):
pois_raw['geometry'] = pois_raw.geometry.centroid

/tmp/ipykernel_553/2648793531.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois_raw['geometry'] = pois_raw.geometry.centroid


In [9]:
# 5: Czyszczenie danych
# Usuwamy puste geometrie i zostawiamy tylko kolumny, których naprawdę potrzebujemy
pois_clean = pois_raw[pois_raw.geometry.notna() & ~pois_raw.geometry.is_empty].copy()

In [11]:
# Wybieramy tylko kluczowe kolumny (jeśli istnieją w pobranym zbiorze)
keep_cols = ["geometry"]
for col in ['amenity', 'highway', 'public_transport', 'name']:
  if col in pois_clean.columns:
    keep_cols.append(col)

pois_clean = pois_clean[keep_cols].reset_index(drop=True)

In [12]:
# 6: Podsumowanie kategorii
print("\nLiczba obiektów według kategorii 'amenity':")
print(pois_clean['amenity'].value_counts(dropna=False))


Liczba obiektów według kategorii 'amenity':
amenity
NaN           4866
restaurant    1986
fast_food     1490
cafe           918
shelter          1
Name: count, dtype: int64


In [13]:
import pandas as pd

# Twoja testowa ramka danych (symuluje brudne dane):
df_test = pd.DataFrame({
    'id': [101, 102, 103, 104, 105],
    'miasto': ['Poznań', 'Warszawa', 'Kraków', 'Gdańsk', 'Wrocław'],
    'populacja': [540000, 1800000, 800000, 470000, 640000],
    'powierzchnia': [262, 517, 327, 262, 293]
}, index=[10, 25, 33, 41, 99]) # celowo dziwne indeksy!

In [14]:
moje_kolumny = []
for col in ['miasto', 'kraj', 'populacja', 'wojewodztwo']:
  if col in df_test:
    moje_kolumny.append(col)

wynik = df_test[moje_kolumny]

In [ ]:
# To robi DOKŁADNIE to samo co Twoja pętla for, ale w jednej linijce - LIST COMPREHENSION:
moje_kolumny = [col for col in ['miasto', 'kraj', 'populacja', 'wojewodztwo'] if col in df_test.columns]

In [15]:
wynik

,miasto,populacja
10,Poznań,540000
25,Warszawa,1800000
33,Kraków,800000
41,Gdańsk,470000
99,Wrocław,640000


In [16]:
df_test = df_test[moje_kolumny].reset_index(drop=True)

In [17]:
df_test

,miasto,populacja
0,Poznań,540000
1,Warszawa,1800000
2,Kraków,800000
3,Gdańsk,470000
4,Wrocław,640000
